# EXP-005 — M4 CPG-GNN: bước 1, TRÍCH CPG bằng Joern (CPU)

Notebook này **không train**. Nó dựng CPG (AST + CFG + data-flow) cho từng hàm
và — quan trọng hơn — **đo throughput thật** để quyết định M4 có đáng làm không.

**Vì sao cài Joern ở đây mà không cài ở máy?** Máy dự án không có Java; Kaggle có
sẵn JDK 17 và quota CPU miễn phí. Cài local sẽ tốn hàng trăm MB, hỏng môi trường,
và không giúp gì — vì **train vẫn phải chạy trên Kaggle**.

**Sửa đúng 1 chỗ:** cell 2 — dòng `REPO = "<user>/<repo>"`.

### Thứ tự chạy (đừng bấm Run All)
1. Cell 2 (clone) → cell 3 (cài JDK + Joern) → **cell 4 (SMOKE — bắt buộc)**
2. Cell 5 (dataset) → cell 6: **đo thử 20 hàm** — xem `ham/giay` thật
3. Cell 7: chạy theo lô cho cả split (dài!) — có `--resume` nên chết giữa chừng
   chạy lại là tiếp tục
4. Cell 8: nạp cache + kiểm tra chất lượng CPG (**gate**: quá mỏng thì dừng M4)
5. **Save Version** → đó là input cho notebook train M4 (bước 2)

### Cell 4 (smoke) là bắt buộc, không phải tuỳ chọn
Máy dự án không có Java nên `scripts/joern/export_cpg.sc` **chưa từng được chạy
thật**. Cell 4 chạy `joern_smoke.sc` trên đúng 1 hàm và in ra danh sách step CPGQL
mà Joern tự khai báo. Nếu cell 4 in được đủ các dòng `ast=`, `callOut=`,
`reachingDef=`… thì exporter chạy được. Nếu không, **dừng lại và gửi lại output
cell 4** — bỏ qua cell này thì cell 6 sẽ chết sau vài phút mà không có thông tin
gì để sửa.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# Cách 1 (khuyên dùng): Kaggle Add-ons → Secrets → GITHUB_TOKEN + HF_TOKEN
# Cách 2: dán trực tiếp dưới đây (ĐỪNG share notebook này)
GITHUB_TOKEN = ""
HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = GITHUB_TOKEN or _s.get_secret("GITHUB_TOKEN")
    HF_TOKEN = HF_TOKEN or _s.get_secret("HF_TOKEN")
except Exception as _e:
    print("[secrets] khong doc duoc secrets:", _e)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
if GITHUB_TOKEN:
    os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")
print("[secrets] github =", bool(GITHUB_TOKEN), "| hf =", bool(HF_TOKEN))

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(f"$ {cmd}\n{r.stdout}{r.stderr}")
    if r.returncode != 0:
        raise RuntimeError(f"lenh that bai (rc={r.returncode}): {cmd}")
    return r.stdout

def py(code, **kw):
    return sh(sys.executable.replace('"', '') + " -c " + json.dumps(code), **kw)


In [ ]:
REPO = "kichirou244/SemanticCodeCloneDetection"
assert REPO != "YOUR_USER/YOUR_REPO", "Sua REPO truoc khi chay cell tiep theo"

if Path(".git").exists():
    print("[repo] da clone o session nay")
else:
    url = f"https://github.com/{REPO}.git"
    if GITHUB_TOKEN:
        url = url.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@")
    sh(f"git clone --depth 1 {url} repo")
    sh("mv repo/* repo/.[!.]* . 2>/dev/null; rmdir repo")
sh("git log --oneline -1 && python -V && nproc && free -g | head -2")

# CPG không đụng torch: notebook này chỉ trích xuất (CPU). torch là của
# notebook train M4 — cài ở đây sẽ tốn thời gian vô ích. Cũng không cài
# torch-geometric: tầng model chưa chạy ở notebook này.
import importlib
need = []
for mod, pkg in [("numpy", "numpy"), ("yaml", "pyyaml"), ("tqdm", "tqdm")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)
print("can cai:", need or "khong")
if need:
    sh("pip install -q " + " ".join(need))
sh("python -c 'import src.representations.cpg.schema as s; print(s.NODE_LABELS); print(s.EDGE_LABELS)'" )


In [ ]:
# CÀI JDK 17 + JOERN — chỉ ở đây (Kaggle), KHÔNG cài máy local.
sh("apt-get update -qq && apt-get install -y -qq openjdk-17-jre-headless")
sh("java -version 2>&1 | head -3")

# Tìm Joern đã có, theo đúng nơi installer chính thức đặt binary. PHẢI thử nhiều
# đường: script cài chọn `~/bin/joern` khi chạy bình thường, nhưng khi notebook
# chạy dưới root thì nó đặt vào `/opt/joern` + symlink vào `/usr/local/bin`.
# Chỉ check một đường là fail giả, rồi tải lại 1,7GB cho nothing.
import shutil

def find_joern():
    for p in (Path.home() / "bin" / "joern", Path("/opt/joern/joern"),
              Path("/usr/local/bin/joern")):
        if p.exists():
            return p
    w = shutil.which("joern")
    return Path(w) if w else None

JOERN_BIN = find_joern()

if JOERN_BIN is None:
    # Chưa có thì mới cài. Dùng script chính thức: tự dò đúng bản theo platform,
    # và KHÔNG hardcode tên file zip vì Joern phát hành bản mới hằng ngày, tên
    # asset đổi theo version (v4.x là joern-cli-linux-x86_64.zip).
    sh("wget -q https://github.com/joernio/joern/releases/latest/download/"
       "joern-install.sh -O /tmp/joern-install.sh && "
       "chmod +x /tmp/joern-install.sh && /tmp/joern-install.sh")
    JOERN_BIN = find_joern()

assert JOERN_BIN is not None, (
    "Khong tim thay joern. Neu vua cai xong, tim bang lenh: "
    "`ls -l /usr/local/bin/joern* /opt/joern/joern*` roi dan duong dan vao JOERN_BIN")
os.environ["JOERN_BIN"] = str(JOERN_BIN)
print("[joern] binary =", JOERN_BIN)
sh(f"{JOERN_BIN} --version 2>&1 | tail -2")

# Gate: phải cả java lẫn joern mới chạy tiếp. In báo cáo rõ ràng thay vì để
# Joern chết âm thầm ở cell sau.
import json as _json
from src.representations.cpg.build import cpg_status
st = cpg_status()
print(_json.dumps(st, indent=2, ensure_ascii=False))
assert st["ready"], "Thieu java/joern — dung cell nay truoc khi chay tiep"

In [ ]:
# SMOKE — chạy 1 hàm qua Joern thật và in output ra đây.
#
# Mục đích: máy local không có Java nên tên step CPGQL trong `export_cpg.sc`
# CHƯA được kiểm chứng. Cell này tốn <1 phút và cho biết ngay tên nào đúng.
# Đọc kết quả:
#   - có dòng "=== METHODS ===" và các dòng ast=/callOut=/reachingDef=...  → OK, đi tiếp
#   - lỗi compile Scala hoặc thiếu dòng nào                            → DỪNG, gửi lại output
sh("python scripts/build_cpg_export.py --config configs/exp005_cpg_gnn.yaml --smoke")

In [ ]:
sh("python scripts/build_dataset.py --config configs/pairs/poj104.yaml")
py("import json; m=json.load(open('data/manifests/poj104.json')); "
   'print("leakage_passed =", m[\"leakage_passed\"]); '
   'print("pairs =", {k:v for k,v in m.get(\"pairs\",{}).items()})')
sh("wc -l data/processed/poj104/samples.jsonl")


In [ ]:
# ĐO THỬ 20 HÀM — đây là phép đo quyết định M4 có đáng làm không.
# Chạy trước cell 7 (lô lớn). Nếu `functions_per_second` thấp hơn ~0,3 hàm/s
# thì 62k hàm > 1 ngày ⇒ cần bàn lại phạm vi trước khi đốt quota Kaggle.
sh("python scripts/build_cpg_export.py --config configs/exp005_cpg_gnn.yaml "
   "--split train --limit 20 --batch-size 20 --out-dir artifacts/cpg_probe")
print(sh("cat artifacts/cpg_probe/benchmark.json"))
sh("head -c 600 $(ls artifacts/cpg_probe/*.json | grep -v benchmark | head -1)")


In [ ]:
# ĐO 500 HÀM — con số dùng để chốt ngân sách giờ cho M4 (thay ước lượng
# 0,3–1 hàm/s trong reports/005). Vài phút, chạy trước khi cam kết cả giờ.
sh("python scripts/build_cpg_export.py --config configs/exp005_cpg_gnn.yaml "
   "--split train --limit 500 --batch-size 100 --out-dir artifacts/cpg_export")
print(sh("cat artifacts/cpg_export/benchmark.json"))


In [ ]:
# Dựng cache .npz từ JSON vừa có (nạp 1 lần, các lần sau chỉ đọc cache).
# max_nodes lấy đúng từ config — đổi số này là đổi feature, cache cũ sẽ sai.
py("from pathlib import Path; import json; "
   'from src.common.config import load_config; '
   'from src.representations.cpg.store import CPGStore; '
   "cfg = load_config(Path('configs/exp005_cpg_gnn.yaml')); "
   "pre = cfg['preprocessing']; "
   "ids = [json.loads(l)['sample_id'] for l in open('data/processed/poj104/samples.jsonl', encoding='utf-8') if json.loads(l)['split']=='train'][:500]; "
   "store = CPGStore.build(Path(pre['export_dir']), ids, pre['max_nodes'], Path(pre['cache_dir'])/'cpp_train500.npz'); "
   'print(json.dumps({k:v for k,v in store.stats.items() if k not in (\"missing\",\"failed\")}, indent=2))')

# GATE CHẤT LƯỢNG CPG — đọc số này trước khi đi tiếp:
#  - nodes_per_code_mean < 20  ⇒ Joern không ra CPG ⇒ dừng M4, sửa exporter
#  - edges_dropped_* lớn        ⇒ nhãn cạnh lệch schema ⇒ sửa schema hoặc exporter
#  - n_missing/n_failed > 5%    ⇒ parse hỏng nhiều ⇒ kiểm tra exporter từng hàm
# Kiểm tra cả export có đủ cả 3 loại cạnh không (AST + CFG + data-flow): nếu
# thiếu REACHING_DEF thì đó không còn là CPG.


### Save Version — đây là bước quan trọng nhất của notebook

Version đã lưu **là input** cho notebook train M4 (đọc `artifacts/cpg_export`
và `artifacts/cpg_cache`). Không Save Version thì mất toàn bộ CPG đã dựng.

**Trước khi Save, gửi lại 4 con số** (chúng quyết định ngân sách M4):
1. output **cell 4 (smoke)** — nếu không có đủ các dòng probe thì M4 chưa chạy được
2. `functions_per_second` (cell 7)
3. `nodes_per_code_mean` (cell 8)
4. `error_rate` (cell 7)

### Chưa làm gì ở bước này (cố ý)
- **Chưa chạy 62k hàm:** ~5–17 h là quyết định ngân sách, không phải việc làm mù.
- **Chưa chạy nhiều tiến trình song song:** giữ nguyên 1 tiến trình để số đo
  đáng tin; song hoá là việc làm sau, khi đã biết số đo một tiến trình.
- **Chưa train GAT:** cần số đo trước (DEC-024).